# AIS Distance Prediction Training

Train an image-based position predictor from the Hugging Face `vision_offset_dataset`.
Data is expected under `ws_aic/data`, and checkpoints are saved under `ws_aic/weight`.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR

PosixPath('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction')

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from model import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_WEIGHT_ROOT,
    VisionOffsetDataset,
    build_resnet_position_model,
    compute_target_stats,
    download_vision_offset_dataset,
    evaluate,
    fit,
    load_samples,
    seed_everything,
    split_samples_by_episode,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

/home/whyz/aic_sejong/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
DATASET_ROOT = DEFAULT_DATASET_ROOT
WEIGHT_ROOT = DEFAULT_WEIGHT_ROOT

CAMERAS = ('center',)  # Use ('left', 'center', 'right') for multi-view training.
TARGET_KEYS = ('x_mm', 'y_mm', 'z_mm')
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'resnet50'
PRETRAINED = True
RUN_NAME = f"vision_offset_{BACKBONE}_{'_'.join(CAMERAS)}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME

(PosixPath('/home/whyz/aic_sejong/ws_aic/data/aic-entrance-dataset/v1.1/vision_offset_dataset'),
 PosixPath('/home/whyz/aic_sejong/ws_aic/weight/ais_distance_prediction/vision_offset_resnet50_center'))

In [4]:
DATASET_ROOT = download_vision_offset_dataset(max_workers=8)

samples = load_samples(DATASET_ROOT)
train_samples, val_samples, test_samples = split_samples_by_episode(
    samples,
    val_ratio=0.2,
    test_ratio=0.0,
    seed=42,
)
target_stats = compute_target_stats(train_samples, TARGET_KEYS)

print(f"samples: train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}")
print('target mean:', target_stats['mean'].tolist())
print('target std :', target_stats['std'].tolist())

samples: train=3520 val=960 test=0
target mean: [1.4557099342346191, 0.07912462204694748, -19.024002075195312]
target std : [5.113223075866699, 2.4719314575195312, 18.063444137573242]


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.08, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=train_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=eval_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['raw_target'][:3]

(torch.Size([16, 3, 224, 224]),
 torch.Size([16, 3]),
 tensor([[  1.7391,   3.4794, -14.7074],
         [  1.6696,   1.6502, -16.3671],
         [  1.5583,   2.3538, -15.7988]]))

In [7]:
model = build_resnet_position_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=len(TARGET_KEYS),
    hidden_dim=256,
    dropout=0.1,
    aggregation='mean',
)
model.to(device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)

24065859

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'cameras': CAMERAS,
    'target_keys': TARGET_KEYS,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    config=config,
)
history[-1]

epochs:   5%|▌         | 1/20 [00:24<07:40, 24.25s/it, train_loss=0.0920, val_euclidean=3.596, val_loss=0.1171]

epoch=001 train_loss=0.0920 val_loss=0.1171 val_mae=1.802 val_euclidean=3.596


epochs:  10%|█         | 2/20 [00:39<05:42, 19.01s/it, train_loss=0.0358, val_euclidean=3.574, val_loss=0.0829]

epoch=002 train_loss=0.0358 val_loss=0.0829 val_mae=1.739 val_euclidean=3.574


epochs:  15%|█▌        | 3/20 [00:54<04:55, 17.36s/it, train_loss=0.0275, val_euclidean=3.541, val_loss=0.0910]

...

epochs:  95%|█████████▌| 19/20 [04:57<00:15, 15.10s/it, train_loss=0.0079, val_euclidean=2.749, val_loss=0.0701]

epoch=019 train_loss=0.0079 val_loss=0.0701 val_mae=1.382 val_euclidean=2.749


epochs: 100%|██████████| 20/20 [05:12<00:00, 15.62s/it, train_loss=0.0076, val_euclidean=2.852, val_loss=0.0776]

epoch=020 train_loss=0.0076 val_loss=0.0776 val_mae=1.414 val_euclidean=2.852


{'epoch': 20.0,
 'train_loss': 0.007605289969466288,
 'val_loss': 0.07762562769154707,
 'val_mae': 1.414133906364441,
 'val_rmse': 1.8052760362625122,
 'val_euclidean': 2.8516845703125}

In [9]:
val_metrics = evaluate(
    model,
    val_loader,
    device,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_metrics

{'loss': 0.07762562769154707,
 'mae': 1.414133906364441,
 'rmse': 1.8052760362625122,
 'euclidean': 2.8516845703125}

In [10]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    pred = model(batch['image'].to(device)).cpu()
pred_mm = pred * target_stats['std'] + target_stats['mean']
preview = torch.cat([pred_mm[:8], batch['raw_target'][:8]], dim=1)
print('columns: pred_x pred_y pred_z target_x target_y target_z')
preview

columns: pred_x pred_y pred_z target_x target_y target_z


tensor([[  2.5294,   0.1374, -11.5097,  -1.4896,   1.2216, -14.4660],
        [  3.1382,  -0.2104, -11.7886,   0.1261,   0.7140, -14.7821],
        [  5.0922,   0.0194, -12.3731,   1.7259,   0.8018, -15.0527],
        [  5.3613,   0.0195, -12.3996,   1.6576,   0.5280, -14.9060],
        [  4.6264,  -0.4163, -12.4006,   1.0537,   0.1431, -14.6456],
        [  4.0978,  -0.4563, -12.4577,   0.6247,   0.0593, -14.5902],
        [  3.9181,  -0.0659, -11.8553,  -0.3926,   2.0627, -14.9651],
        [  3.3501,   0.8173, -12.1889,  -0.9627,   4.1000, -15.1820]])